Spark ships with hundreds of built-in SQL functions that run entirely on the JVM — no Python serialization, no overhead. When a built-in cannot express your logic, User-Defined Functions (UDFs) let you bring Python code into the pipeline. This notebook covers the most useful built-in functions by category, then Python UDFs and the faster Pandas UDF (vectorized) alternative.

## Function Performance Tiers

![UDF Performance](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-udf-performance.svg)

Always exhaust built-in functions before writing a UDF. A plain Python UDF serializes each row across the JVM–Python boundary twice — it can be 10–100× slower than an equivalent built-in.

## Setup

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("SQLFunctionsUDFs")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

DATA = os.path.join(os.path.dirname(os.path.abspath("11-sql-functions-udfs.ipynb")), "data")

customers = spark.read.option("header","true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),       False),
    StructField("card_id",     StringType(),       False),
    StructField("customer_id", StringType(),       False),
    StructField("amount",      DecimalType(18,2),  False),
    StructField("merchant",    StringType(),       True),
    StructField("category",    StringType(),       True),
    StructField("txn_ts",      TimestampType(),    False),
    StructField("status",      StringType(),       False),
    StructField("is_fraud",    BooleanType(),      False),
])).parquet(f"{DATA}/card_transactions")

loan_accounts = spark.read.option("multiLine","true").schema(StructType([
    StructField("loan_id",       StringType(),       False),
    StructField("customer_id",   StringType(),       False),
    StructField("loan_type",     StringType(),       False),
    StructField("principal",     DecimalType(18,2),  False),
    StructField("interest_rate", DoubleType(),       False),
    StructField("tenure_months", IntegerType(),      False),
    StructField("disbursed_on",  DateType(),         False),
    StructField("status",        StringType(),       False),
])).json(f"{DATA}/loan_accounts")

payments = spark.read.format("delta").load(f"{DATA}/payments")

# Register as views for SQL examples
customers.createOrReplaceTempView("customers")
card_txns.createOrReplaceTempView("card_transactions")
loan_accounts.createOrReplaceTempView("loan_accounts")
payments.createOrReplaceTempView("payments")

print("Tables loaded and registered as views.")

## String Functions

Beyond `upper`, `lower`, and `split` covered in earlier notebooks, here are the string functions most useful in financial data pipelines.

In [ ]:
from pyspark.sql.functions import (
    col, lpad, rpad, format_string, instr, locate,
    regexp_extract, regexp_replace, translate, overlay,
    repeat, reverse, sentences
)

# lpad / rpad — pad account numbers and codes to fixed width
customers.select(
    "customer_id",
    lpad(col("customer_id"), 10, "0").alias("padded_id"),      # "CUST0001" → "00CUST0001"
    rpad(col("city"), 12, ".").alias("city_padded"),            # "Mumbai......"
).show(5)

In [ ]:
# format_string — build formatted display strings (like Python f-strings, JVM-side)
customers.select(
    format_string("%-20s | %s, %s", col("full_name"), col("city"), col("state"))
    .alias("display_label")
).show(6, truncate=False)

In [ ]:
# regexp_extract — pull structured data from free-text fields
# Extract the numeric part of customer_id ("CUST0001" → "0001")
customers.select(
    "customer_id",
    regexp_extract(col("customer_id"), r"CUST(\d+)", 1).alias("cust_num"),
    regexp_extract(col("email"), r"@(.+)", 1).alias("email_domain"),
).show(5)

In [ ]:
# translate — character-level substitution (useful for data masking)
# Replace digits in a simulated card number with *
card_txns.select(
    "txn_id",
    col("card_id"),
    translate(col("card_id"), "0123456789", "**********").alias("masked_card_id"),
).show(5)

## Numeric Functions

| Function | Purpose |
|---|---|
| `abs` | Absolute value |
| `ceil` / `floor` | Round up / down to nearest integer |
| `round(col, n)` | Round to n decimal places |
| `bround(col, n)` | Banker's rounding (round half to even) |
| `pow(col, n)` | Raise to power |
| `sqrt` | Square root |
| `log(base, col)` | Logarithm |
| `greatest(*cols)` | Row-wise maximum across columns |
| `least(*cols)` | Row-wise minimum across columns |
| `percent_rank()` | Relative rank as 0.0–1.0 (window function) |

In [ ]:
from pyspark.sql.functions import (
    abs as _abs, ceil, floor, round as _round,
    pow as _pow, sqrt, log, greatest, least, col
)

card_txns.select(
    "txn_id",
    col("amount"),
    _round(col("amount"), 0).alias("amount_rounded"),
    ceil(col("amount") / 1000).alias("amount_k_ceil"),         # threshold buckets
    floor(col("amount") / 1000).alias("amount_k_floor"),
    _round(col("amount") * 0.18, 2).alias("gst_18pct"),        # 18% GST
    greatest(col("amount"), lit(100)).alias("min_charge"),      # floor at ₹100
).show(5)

In [ ]:
from pyspark.sql.functions import lit

# Compound interest calculation — total repayment on a loan
# A = P * (1 + r/n)^(n*t)  simplified to annual compounding
loan_accounts.select(
    "loan_id",
    "principal",
    "interest_rate",
    "tenure_months",
    _round(
        col("principal") * _pow(
            lit(1) + col("interest_rate") / lit(100),
            col("tenure_months") / lit(12)
        ), 2
    ).alias("total_repayment"),
).show(5)

## Date & Time Functions — Advanced

Building on `year`, `month`, `datediff` from the Core Transformations notebook, here are the more advanced date functions used in financial reporting.

In [ ]:
from pyspark.sql.functions import (
    date_trunc, trunc, last_day, next_day,
    months_between, date_format, current_date,
    quarter, weekofyear, unix_timestamp, from_unixtime
)

card_txns.select(
    "txn_id",
    col("txn_ts"),
    date_trunc("month",  col("txn_ts")).alias("month_start"),   # first moment of the month
    date_trunc("week",   col("txn_ts")).alias("week_start"),
    last_day(col("txn_ts").cast("date")).alias("month_end"),
    quarter(col("txn_ts")).alias("quarter"),
    weekofyear(col("txn_ts")).alias("iso_week"),
    date_format(col("txn_ts"), "EEEE").alias("day_of_week"),    # "Monday", "Tuesday"…
).show(5)

In [ ]:
# months_between — tenure elapsed for active loans
from pyspark.sql.functions import months_between, current_date, round as _round, col

loan_accounts.select(
    "loan_id",
    "disbursed_on",
    "tenure_months",
    _round(months_between(current_date(), col("disbursed_on")), 1).alias("months_elapsed"),
    _round(
        months_between(current_date(), col("disbursed_on")) / col("tenure_months") * 100, 1
    ).alias("pct_complete"),
).show(6)

## Conditional Functions

| Function | Equivalent SQL | Notes |
|---|---|---|
| `when(cond, val).otherwise(val)` | `CASE WHEN` | Covered in Core Transformations |
| `coalesce(*cols)` | `COALESCE` | First non-null value |
| `nullif(col, val)` | `NULLIF` | Returns null if col == val |
| `nanvl(col, replacement)` | — | Replace NaN with a value (numeric columns) |
| `isnull` / `isnotnull` | `IS NULL` | Null check |

In [ ]:
from pyspark.sql.functions import coalesce, nullif, lit, col

# coalesce — fallback chain: use email, else phone, else "NO_CONTACT"
customers.select(
    "customer_id",
    "full_name",
    coalesce(col("email"), col("phone"), lit("NO_CONTACT")).alias("contact"),
).show(6, truncate=False)

## Array Functions

When a column holds an `ArrayType`, these functions operate element-wise without needing a UDF.

| Function | Purpose |
|---|---|
| `array(*cols)` | Build an array from columns |
| `size(col)` | Array length |
| `array_contains(col, val)` | Check membership |
| `explode(col)` | One row per array element |
| `array_distinct(col)` | Remove duplicates |
| `array_sort(col)` | Sort elements |
| `array_union` / `array_intersect` | Set operations |
| `flatten` | Flatten nested arrays |

In [ ]:
from pyspark.sql.functions import (
    array, size, array_contains, explode,
    array_distinct, array_sort, collect_list, col
)

# Build a product array per customer — which products does each customer hold?
product_arrays = (
    card_txns
    .filter(col("category").isNotNull())
    .groupBy("customer_id")
    .agg(
        array_distinct(collect_list("category")).alias("categories_used"),
        array_distinct(collect_list("status")).alias("txn_statuses"),
    )
)
product_arrays.show(5, truncate=False)

In [ ]:
# explode — one row per category per customer (inverse of collect_list)
from pyspark.sql.functions import explode, col

product_arrays.select(
    "customer_id",
    explode("categories_used").alias("category"),
).show(10)

## Higher-Order Functions

Higher-order functions apply a lambda directly to array elements on the JVM — no UDF serialization overhead.

| Function | Purpose |
|---|---|
| `transform(col, func)` | Map over array elements |
| `filter(col, func)` | Keep elements matching condition |
| `aggregate(col, zero, merge)` | Fold array to a scalar |
| `exists(col, func)` | True if any element matches |
| `forall(col, func)` | True if all elements match |

In [ ]:
from pyspark.sql.functions import transform, filter as array_filter, aggregate, exists, col, lit

# transform — convert category names to uppercase inside the array
product_arrays.select(
    "customer_id",
    transform("categories_used", lambda x: x.alias("cat")).alias("cats_raw"),
).show(3, truncate=False)

# Cleaner: use SQL expression via selectExpr for lambda syntax
product_arrays.selectExpr(
    "customer_id",
    "transform(categories_used, x -> upper(x))          AS categories_upper",
    "filter(txn_statuses, s -> s != 'REVERSED')         AS clean_statuses",
    "exists(txn_statuses, s -> s = 'DECLINED')          AS had_decline",
    "aggregate(array(1,2,3,4,5), 0, (acc, x) -> acc+x) AS sum_demo",
).show(5, truncate=False)

## Python UDFs

A UDF wraps a Python function so it can be called inside `select` or `spark.sql`. Use `@udf(returnType)` as a decorator for the DataFrame API, or `spark.udf.register(name, func, returnType)` to make it available in SQL.

**Important:** every Python UDF serializes each row across the JVM–Python boundary. This is significantly slower than built-in functions. Always check if a built-in can do the job first.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# UDF 1 — Classify transaction amount into buckets
# (could be done with when/otherwise, shown here to illustrate UDF pattern)
def amount_bucket(amount) -> str:
    if amount is None:
        return "UNKNOWN"
    amt = float(amount)
    if amt > 20000: return "LARGE"
    if amt > 5000:  return "MEDIUM"
    if amt > 500:   return "SMALL"
    return "MICRO"

amount_bucket_udf = udf(amount_bucket, StringType())

card_txns.select(
    "txn_id",
    "amount",
    amount_bucket_udf(col("amount")).alias("bucket"),
).show(8)

In [ ]:
# UDF 2 — Mask PAN (card number) — keep first 6 and last 4, mask middle digits
# Real card numbers follow: 4111 1111 1111 1111 — first 6 = BIN, last 4 = visible
import re

def mask_pan(card_id: str) -> str:
    """Mask a card identifier: keep first 4 chars and last 2, mask the middle."""
    if card_id is None:
        return None
    # card_id format: "CARD0042" — mask middle digits
    return re.sub(r"(\w{4})(\w+)(\w{2})", r"\1****\3", card_id)

mask_pan_udf = udf(mask_pan, StringType())

card_txns.select(
    "card_id",
    mask_pan_udf(col("card_id")).alias("masked_card_id"),
    "amount",
    "status",
).show(8)

In [ ]:
# Register for SQL use — same UDF callable from spark.sql()
spark.udf.register("amount_bucket", amount_bucket, StringType())
spark.udf.register("mask_pan",      mask_pan,      StringType())

spark.sql("""
    SELECT
        mask_pan(card_id)          AS masked_card,
        customer_id,
        amount,
        amount_bucket(amount)      AS bucket,
        status
    FROM   card_transactions
    WHERE  status = 'SUCCESS'
    ORDER  BY amount DESC
    LIMIT  10
""").show()

## Pandas UDFs — Vectorized UDFs

Pandas UDFs (also called vectorized UDFs) use Apache Arrow to transfer data between JVM and Python as columnar batches — far more efficient than row-by-row Python UDFs.

Three types:
| Type | Input | Output | Use case |
|---|---|---|---|
| `SCALAR` | `pd.Series` per column | `pd.Series` | Row-wise transformation |
| `SCALAR_ITER` | Iterator of `pd.Series` | Iterator of `pd.Series` | Setup once, process batches |
| `GROUP_MAP` | `pd.DataFrame` per group | `pd.DataFrame` | Custom aggregation |

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType, DoubleType

# Pandas UDF — vectorized amount bucket classification
@pandas_udf(StringType())
def amount_bucket_vec(amounts: pd.Series) -> pd.Series:
    def classify(amt):
        if pd.isna(amt): return "UNKNOWN"
        if amt > 20000:  return "LARGE"
        if amt > 5000:   return "MEDIUM"
        if amt > 500:    return "SMALL"
        return "MICRO"
    return amounts.apply(classify)

# Pandas UDF — calculate EMI using standard formula
@pandas_udf(DoubleType())
def calc_emi(principal: pd.Series, rate: pd.Series, tenure: pd.Series) -> pd.Series:
    """Monthly EMI = P * r * (1+r)^n / ((1+r)^n - 1)  where r = monthly rate"""
    r = rate / 100 / 12          # monthly interest rate
    n = tenure.astype(float)
    return (principal.astype(float) * r * (1 + r)**n / ((1 + r)**n - 1)).round(2)

# Apply to DataFrames
card_txns.select(
    "txn_id", "amount",
    amount_bucket_vec(col("amount")).alias("bucket"),
).show(8)

loan_accounts.select(
    "loan_id", "principal", "interest_rate", "tenure_months",
    calc_emi(col("principal"), col("interest_rate"), col("tenure_months")).alias("monthly_emi"),
).show(6)

## When to Use Each

| Scenario | Recommended approach |
|---|---|
| Case/when logic, string ops, date math | Built-in `pyspark.sql.functions` |
| Complex ML feature engineering on numbers | `@pandas_udf(DoubleType())` |
| Calling an external library (scikit-learn, spaCy) | `@pandas_udf` with `SCALAR_ITER` |
| Custom aggregation needing full pandas logic | `@pandas_udf` `GROUP_MAP` |
| Simple logic with no built-in alternative | `@udf` — acceptable for small data |
| `@udf` on billions of rows | ❌ Avoid — use Pandas UDF or rewrite with builtins |

## Summary

- Spark ships with hundreds of built-in functions in `pyspark.sql.functions` — always prefer them over UDFs; they run on the JVM with no serialization overhead.
- String: `lpad`, `format_string`, `regexp_extract`, `regexp_replace`, `translate` cover most data masking and formatting needs.
- Numeric: `ceil`, `floor`, `round`, `pow`, `greatest`, `least` handle financial calculations.
- Date/time: `date_trunc`, `last_day`, `months_between`, `quarter`, `date_format` handle reporting periods.
- Array functions (`collect_list`, `explode`, `array_distinct`) and higher-order functions (`transform`, `filter`, `aggregate` via `selectExpr`) process arrays without UDFs.
- Python UDFs (`@udf`) cross the JVM–Python boundary per row — 10–100× slower than built-ins.
- Pandas UDFs (`@pandas_udf`) use Apache Arrow to process batches — significantly faster than plain UDFs.
- Register UDFs with `spark.udf.register(name, func, returnType)` to call them from `spark.sql(...)`.